In [ ]:
# Linalg libraries
import numpy as np

# Plotting libraries
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# Data management libraries
import h5py as hf
from dataclasses import dataclass
import glob

In [ ]:
@dataclass
class ModelTrajectory:
    # Model data
    width: int
    depth: int
    activation: str
    lr: float
    architecture: str
    ds_size: float

    # Loss data
    train_loss: np.ndarray
    train_loss_std: np.ndarray
    test_loss: np.ndarray
    test_loss_std: np.ndarray

    # CV data
    entropy: np.ndarray
    entropy_std: np.ndarray
    trace: np.ndarray
    trace_std: np.ndarray

In [ ]:
def load_model_trajectory(file_root, file_signature, accuracy = False):
    # Glob all files with the correct signature
    files = glob.glob(file_root + '/*' + file_signature + '*.h5')

    # Extract model parameters
    model_params = files[0].split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    ds_size = float(model_params[5])    
    lr = float(model_params[7])
    architecture = model_params[1]

    # Load data
    train_losses = []
    test_losses = []

    if accuracy:
        train_accuracies = []
        test_accuracies = []
    
    entropies = []
    traces = []

    for file in files:
        with hf.File(file, 'r') as f:
            if file.split("/")[-1].split("_")[0] == "train":
                train_losses.append(f['loss'][:])
                if accuracy:
                    train_accuracies.append(f['accuracy'][:])
                entropies.append(f['entropy'][:])
                traces.append(f['trace'][:])
            else:
                test_losses.append(f['loss'][:])
                if accuracy:
                    test_accuracies.append(f['accuracy'][:])

    # Take the means, std and build the ModelTrajectory object
    train_loss = np.nanmean(train_losses, axis=0)
    test_loss = np.nanmean(test_losses, axis=0)
    train_loss_std = np.nanstd(train_losses, axis=0)
    test_loss_std = np.nanstd(test_losses, axis=0)

    if accuracy:
        train_accuracy = np.mean(train_accuracies, axis=0)
        test_accuracy = np.mean(test_accuracies, axis=0)

        train_accuracy_std = np.std(train_accuracies, axis=0)
        test_accuracy_std = np.std(test_accuracies, axis=0)

    entropy = np.nanmean(entropies, axis=0)
    trace = np.nanmean(traces, axis=0)
    entropy_std = np.std(entropies, axis=0)
    trace_std = np.std(traces, axis=0)

    return ModelTrajectory(
        width=width,
        depth=depth,
        activation=activation,
        lr=lr,
        architecture=architecture,
        ds_size=ds_size,
        train_loss=train_loss,
        train_loss_std=train_loss_std,
        test_loss=test_loss,
        test_loss_std=test_loss_std,
        entropy=entropy,
        entropy_std=entropy_std,
        trace=trace,
        trace_std=trace_std
    )

def load_model_trajectories(root_path):
    files = glob.glob(root_path + '/*.h5')

    # Get unique names
    unique_names = []
    for file in files:
        unique_names.append(
            "_".join(file.split('/')[-1].split('_')[2:7])
            )

    unique = np.unique(unique_names)

    # Load data
    data = []
    for name in unique:
        data.append(load_model_trajectory(root_path, name))

    return data   

In [ ]:
ce_data = load_model_trajectories('ce/')
mse_2_data = load_model_trajectories('mse-2/')
mse_4_data = load_model_trajectories('mse-4/')

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(25, 25))

activations = ["relu", "tanh", "sigmoid"]
line_styles = ['-', '--']
colours = ['b', 'r', 'g', 'y']


# CE data
for i, model in enumerate(ce_data):
    ax[0, 0].plot(model.train_loss, color=colours[activations.index(model.activation)])
    # ax[row, 0].fill_between(
    #     np.arange(len(model.train_loss)),
    #     model.train_loss - model.train_loss_std,
    #     model.train_loss + model.train_loss_std,
    #     alpha=0.3
    # )
    ax[0, 0].set_yscale("log")

for i, model in enumerate(ce_data):
    ax[0, 1].plot(model.trace, color=colours[activations.index(model.activation)])
    # ax[row, 1].fill_between(
    #     np.arange(len(model.trace)),
    #     model.train_loss - model.trace_std,
    #     model.train_loss + model.trace_std,
    #     alpha=0.3
    # )
    ax[0, 1].set_yscale("log")


for i, model in enumerate(ce_data):
    ax[0, 2].plot(model.entropy, color=colours[activations.index(model.activation)])
    # ax[row, 2].fill_between(
    #     np.arange(len(model.entropy)),
    #     model.train_loss - model.entropy_std,
    #     model.train_loss + model.entropy_std,
    #     alpha=0.3
    # )

# MSE-2 data
for i, model in enumerate(mse_2_data):
    ax[1, 0].plot(model.train_loss, color=colours[activations.index(model.activation)])
    # ax[row, 0].fill_between(
    #     np.arange(len(model.train_loss)),
    #     model.train_loss - model.train_loss_std,
    #     model.train_loss + model.train_loss_std,
    #     alpha=0.3
    # )
    ax[1, 0].set_yscale("log")

for i, model in enumerate(mse_2_data):
    ax[1, 1].plot(model.trace, color=colours[activations.index(model.activation)])
    # ax[row, 1].fill_between(
    #     np.arange(len(model.trace)),
    #     model.train_loss - model.trace_std,
    #     model.train_loss + model.trace_std,
    #     alpha=0.3
    # )
    ax[1, 1].set_yscale("log")


for i, model in enumerate(mse_2_data):
    ax[1, 2].plot(model.entropy, color=colours[activations.index(model.activation)])
    # ax[row, 2].fill_between(
    #     np.arange(len(model.entropy)),
    #     model.train_loss - model.entropy_std,
    #     model.train_loss + model.entropy_std,
    #     alpha=0.3
    # )

# MSE-4 data
for i, model in enumerate(mse_4_data):
    ax[2, 0].plot(model.train_loss, color=colours[activations.index(model.activation)])
    # ax[row, 0].fill_between(
    #     np.arange(len(model.train_loss)),
    #     model.train_loss - model.train_loss_std,
    #     model.train_loss + model.train_loss_std,
    #     alpha=0.3
    # )
    ax[2, 0].set_yscale("log")

for i, model in enumerate(mse_4_data):
    ax[2, 1].plot(model.trace, color=colours[activations.index(model.activation)])
    # ax[row, 1].fill_between(
    #     np.arange(len(model.trace)),
    #     model.train_loss - model.trace_std,
    #     model.train_loss + model.trace_std,
    #     alpha=0.3
    # )
    ax[2, 1].set_yscale("log")

for i, model in enumerate(mse_4_data):
    ax[2, 2].plot(model.entropy, color=colours[activations.index(model.activation)])
    # ax[row, 2].fill_between(
    #     np.arange(len(model.entropy)),
    #     model.train_loss - model.entropy_std,
    #     model.train_loss + model.entropy_std,
    #     alpha=0.3
    # )

red_patch = mpatches.Patch(color='red', label='ReLU')
blue_patch = mpatches.Patch(color='blue', label='Tanh')
green_patch = mpatches.Patch(color='green', label='Sigmoid')


ax[0, 1].legend(handles=[red_patch, blue_patch, green_patch], title="Activations")


# Add title to first row of train loss, trace, entropy
ax[0, 0].set_title("Train Loss")
ax[0, 1].set_title("Trace")
ax[0, 2].set_title("Entropy")

# Add x label to bottom row of epoch
ax[-1, 0].set_xlabel("Epoch")
ax[-1, 1].set_xlabel("Epoch")
ax[-1, 2].set_xlabel("Epoch")

ax[0, 0].set_ylabel("CE")
ax[1, 0].set_ylabel("MSE-2")
ax[2, 0].set_ylabel("MSE-4")




plt.savefig("perceptron.pdf")
plt.show()
